In [3]:
import pandas as pd
import requests
import os
from urllib.parse import urlparse
from tqdm import tqdm  


INPUT_CSV = r"C:\Users\bryso\Documents\Work\UGA\observations-813791.csv\observations-813791.csv"         # path to existing CSV
OUTPUT_CSV = r"D:\Project data\unlabeled_data.csv"       # path where the new CSV will be saved
IMAGE_SAVE_DIR = r"D:\Project data\unlabeled_images"  # folder where images will be saved
URL_COLUMN = 'image_url'                # column name containing the image URLs

# read the original CSV
df = pd.read_csv(INPUT_CSV)

COLUMNS_TO_KEEP = [col for col in df.columns if col != URL_COLUMN] # retain other columns in csv file

# create the image destination folder if it doesn't already exist
os.makedirs(IMAGE_SAVE_DIR, exist_ok=True)

# list to hold the rows for the new CSV
new_data = []

# wrap df.iterrows() with tqdm to generate the progress bar
for index, row in tqdm(df.iterrows(), total=len(df), desc="Downloading Images"):
    image_url = row[URL_COLUMN]
    
    # extract only the specified columns for the new row
    row_data = {col: row[col] for col in COLUMNS_TO_KEEP}
    
    # check if the URL is valid/exists
    if pd.notna(image_url) and str(image_url).strip() != "":
        try:
            # extract the original file extension (e.g., .jpg, .png)
            parsed_url = urlparse(image_url)
            original_filename = os.path.basename(parsed_url.path)
            _, ext = os.path.splitext(original_filename)
            
            # if the URL doesn't have an extension then default to .jpg
            if not ext:
                ext = ".jpg"
            
            # create the new filename based on the row number
            filename = f"unlabeled_image_{index + 1:02d}{ext}"
                
            local_path = os.path.join(IMAGE_SAVE_DIR, filename)
            
            # download the image
            response = requests.get(image_url, stream=True, timeout=10)
            response.raise_for_status() # Raises an error for bad HTTP status codes
            
            # save the image to the specified location
            with open(local_path, 'wb') as file:
                for chunk in response.iter_content(chunk_size=8192):
                    file.write(chunk)
                    
            # add the new local file path to the row data
            row_data['local_image_path'] = local_path
            
            
        except Exception as e:
            
            # using tqdm.write instead of print prevents breaking the progress bar visually
            tqdm.write(f"Failed to download row {index} ({image_url}): {e}")
            row_data['local_image_path'] = 'Download Error'
    else:
        row_data['local_image_path'] = 'No URL Provided'
        
    new_data.append(row_data)

# generate and save the new CSV
new_df = pd.DataFrame(new_data)
new_df.to_csv(OUTPUT_CSV, index=False)

print(f"\nProcess complete! New CSV saved to: {OUTPUT_CSV}")


Process complete! New CSV saved to: D:\Project data\unlabeled_data.csv
